
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Beyond Prompts – Retrieval Agents and Context Engineering

## Introduction

As Generative AI applications move from prototypes to production, developers often encounter a performance ceiling. No matter how cleverly we rewrite instructions, we cannot force a model to know facts it was never trained on or prevent it from hallucinating when it lacks critical data. This marks a fundamental transition—from **Prompt Engineering**, which focuses on crafting the query, to **Context Engineering**, which focuses on architecting the entire environment supplied to the model. In this lesson, we'll explore this evolution, defining the boundaries of prompting, introducing the architectural solution of **Retrieval Augmented Generation (RAG)**, and examining the advanced discipline of engineering the context window for reliability and cost-efficiency.

## Lesson Objectives

- Distinguish between prompt engineering and context engineering based on scope and goals
- Identify the three critical failure modes of prompting that necessitate RAG architectures
- Explain the challenges of RAG architectures and strategies to address them
- Define the fundamentals of context engineering and its importance for agentic systems
- Apply the main principles of context engineering to optimize model performance
- Implement strategies to manage cost, latency, and performance of agentic systems

**Note on Course Scope:** This lesson introduces Context Engineering concepts for architectural awareness. The hands-on exercises will focus specifically on foundational RAG implementation.

## A. Foundations of Prompt Engineering

Before we can build complex AI architectures, we must first master the fundamental unit of interaction: the prompt. In this section, we'll explore the art of crafting instructions to guide model behavior, distinguishing between statistical prediction and genuine reasoning capabilities. However, as we refine our prompts, we'll inevitably encounter the hard boundaries of what a model can know based solely on its training data.

### A1. Defining Prompt Engineering

Prompt Engineering is the practice of refining the input text (the prompt) to optimize the output generated by a Large Language Model (LLM). It's a **tactical** discipline focused on the instruction layer. We use techniques such as **Few-Shot Prompting** (providing examples) and **Persona Adoption** (assigning a role) to guide the model's behavior and formatting. Ideally, prompt engineering treats the model as a reasoning engine, guiding it to leverage its pretrained weights to solve problems effectively.

### A2. Reasoning vs. Non-Reasoning Models

Effective prompting requires understanding the capabilities of the underlying model. Let's examine the two primary categories:

- **Non-Reasoning Models (e.g., Llama 3, GPT-4o):** These models predict the next token based on statistical likelihood. They require explicit guidance, such as **Chain of Thought (CoT)** prompting ("Think step-by-step"), to break down complex logic and avoid rushing to incorrect conclusions.
  
- **Reasoning Models (e.g., OpenAI o1):** These models are trained to generate their own internal chain of thought before producing a final answer. For these models, manual CoT prompting is often redundant or counterproductive. Context engineering for reasoning models focuses on defining the *goal* and *constraints* rather than the *thinking process*.

### A3. The Boundaries of Prompting

Prompt engineering has hard boundaries. No amount of instruction refinement can overcome the following limitations inherent to the model's training data:

1. **The Knowledge Cutoff:** The model cannot answer questions about events that occurred after the cutoff date of its training data.
   - *Example Prompt:* `Who won the 2025 Nobel Prize in Physics?`

1. **Hallucination:** When asked for specific facts without external references, models often prioritize plausibility over truth, fabricating citations or data points.
   - *Example Prompt:* `Find a scientific reference proving that avocado reduces blood sugar levels`

1. **Ambiguity:** Without private context, models default to generic interpretations.
   - *Example Prompt:* `Explain how to secure a lakehouse.` (This triggers advice on physical home security rather than Databricks Data Lakehouse governance.)

## B. Retrieval Augmented Generation (RAG)

While prompt engineering optimizes *how* a model responds, it cannot address the fundamental issue of *what* a model knows. To overcome the limitations of frozen training data and hallucination, we must shift our architectural approach from relying on internal memory to leveraging external context. In this section, we'll introduce Retrieval Augmented Generation (RAG), the critical framework that bridges the gap between a model's pretrained knowledge and your proprietary data.

**RAG vs. Retrieval Agent:** 
RAG refers to the architectural pattern of coupling retrieval with generation, independent of any specific tooling or agent logic. **Retrieval agents**, by contrast, are concrete implementations that operationalize this pattern—handling query routing, retrieval orchestration, and context assembly within real systems.

### B1. Defining RAG

To solve the knowledge boundaries we defined earlier, we shift from a memory-based approach to a context-based architecture known as **Retrieval Augmented Generation (RAG)**. RAG injects proprietary or real-time data into the prompt, enabling the model to respond based on provided facts rather than its internal memory.

The RAG process consists of three key stages:

- **Retrieval:** The system searches a knowledge base (indexed via **Mosaic AI Vector Search**) for relevant data chunks
- **Augmentation:** The system injects these chunks into the context window
- **Generation:** The model synthesizes an answer using *only* the injected data

### B2. The Context Challenge

While RAG solves the knowledge gap, it introduces a new challenge: **Context Rot**. Early RAG implementations often failed because developers would simply retrieve large volumes of documents and paste them into the prompt. This approach overwhelms the model, leading to:

- **Context Poisoning:** The inclusion of irrelevant or conflicting information that confuses the model
- **Lost in the Middle:** The tendency of models to ignore information buried in the middle of a long context window, prioritizing data at the very beginning or very end

These challenges highlight the need for strategic context management, which we'll explore in the next section.

## C. Principles of Context Engineering

Simply retrieving data is not enough; dumping raw information into a prompt often leads to confusion rather than clarity. As we move from basic RAG to production-grade systems, we must treat the context window not as a passive container, but as an engineered environment that actively shapes model behavior. In this section, we'll outline the principles of Context Engineering, emphasizing how to structure, filter, and ground information to ensure reliability and accuracy.

### C1. Defining the Context Environment

**Context Engineering** is the strategic design of the entire input window. It moves beyond writing a single instruction to managing the complete system state. We orchestrate the interplay between the **System Instructions**, **Conversation History**, **Retrieved Data**, and **User Constraints** to ensure the model has exactly the signal it needs to perform the task effectively.

### C2. Designing System Prompts

In a Context Engineering paradigm, the System Prompt is not just a request—it's a behavioral program that defines how the model should operate.

Key components of an effective system prompt include:

- **Role Definition:** Explicitly define the persona (e.g., "You are a Databricks Security Architect")
- **Negative Constraints:** Define what the model *cannot* do (e.g., "Do not mention competitor products" or "Do not provide code unless explicitly requested")
- **Output Formatting:** Enforce structured outputs (e.g., JSON, YAML, or Markdown tables) to ensure downstream applications can parse the response deterministically

### C3. Strict Grounding and Chunking

Retrieval must be strictly governed to prevent hallucinations and ensure accuracy.

Key strategies include:

- **Grounding Instructions:** Use explicit instructions to bind the model to the retrieved context. For example: *"Answer using ONLY the provided context chunks. If the answer is not present, state 'I do not have that information.'"*
- **Metadata Filtering:** Utilize **Unity Catalog** metadata to filter retrieval *before* it reaches the model. For example, if a user asks about "2024 Revenue," the system should filter chunks where `year=2024` to prevent the model from seeing outdated 2023 data

### C4. Managing Multi-Turn State

For applications involving long conversations (such as Agents), the context window will eventually fill up. Context Engineering requires strategic approaches to manage this state:

- **Summarization:** Periodically compress the conversation history into a summary of key decisions and facts
- **Moving Window:** Discard the oldest messages to free up token space for new retrieval
- **Selective Persistence:** Determine which pieces of information (e.g., user name, current project ID) must remain in the context permanently versus what can be discarded

These strategies ensure that we maintain relevant context while staying within token limits.

## D. Context Constraints: Token Budgets and Window Limits

Even the most elegantly engineered context is subject to the hard constraints of compute resources and model architecture. As we scale our applications, we must balance the desire for comprehensive context against the realities of token limits, latency, and operational costs. In this section, we'll examine the economics of the context window, providing strategies to optimize token budgets without sacrificing response quality.

### D1. Understanding Context Windows

Every model has a **Context Window Limit** (e.g., 8k, 32k, or 128k tokens). This represents the hard limit of working memory available to the model.

The context window consists of:

- **Input Tokens:** The text we send to the model (instructions + retrieved documents + history)
- **Output Tokens:** The text the model generates

**The Trade-off:** As we fill the window with more retrieved data, the model's ability to reason degrades (the "Lost in the Middle" phenomenon), and latency increases significantly. Strategic context management is essential for maintaining performance.

### D2. Token Economics and Optimization

Context is not free. **Databricks Foundation Model APIs** (and other providers) charge based on the volume of input and output tokens consumed.

Key considerations:

- **Cost Management:** A naive RAG system that retrieves 50 documents for every query will burn through token budgets rapidly

**Optimization Strategies:**

- **Just-in-Time Retrieval:** Instead of loading a full manual at the start of a chat, give the Agent a tool to retrieve specific sections only when the user asks a relevant question
- **Reranking:** Use a reranker model to score the top 50 retrieved chunks and only inject the top 3-5 most relevant ones into the final context window

These strategies help us balance comprehensive context with cost efficiency and performance.

## E. Summary

Moving from Prompt Engineering to Context Engineering represents a fundamental shift from "micro-optimization" to "macro-architecture." While prompts control tone and format, they cannot bridge the knowledge gap inherent in LLMs. RAG architectures solve this by injecting external data, but introduce complexity around context window management. Context Engineering addresses these challenges by rigorously structuring the input, enforcing grounding rules, and managing token budgets to create reliable, cost-effective AI systems.

**Key Takeaways:**

1. **Retrieval Agent Architecture:** Use retrieval components to bridge knowledge gaps, and apply Context Engineering to optimize retrieval agents for performance, reliability, and cost.
2. **Context is Finite:** Manage the context window like a budget. Use filtering and reranking to maximize the value of every token.
3. **Grounding is Mandatory:** Strictly instruct the model to use *only* retrieved data and leverage **Unity Catalog** metadata to ensure that data is relevant and secure.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>